In [13]:
import pandas as pd
import numpy as np

INCLUDE_FACE = False
INCLUDE_DEPTH = False

In [16]:
def generate_full_column_list():
    """
    Generates the standardized list of 1629 column names (x/y/z for 543 landmarks).
    """
    landmark_specs = {
        'face': 468,  # Indices 0 to 467
        'left_hand': 21, # Indices 0 to 20
        'pose': 33,    # Indices 0 to 32
        'right_hand': 21, # Indices 0 to 20
    }

    if not INCLUDE_FACE:
        del landmark_specs['face']

    axes = ['x', 'y', 'z'] if INCLUDE_DEPTH else ['x', 'y']
    
    full_columns = []
    
    for landmark_type, count in landmark_specs.items():
        for i in range(count):
            for axis in axes:
                full_columns.append(f"{landmark_type}_{i}_{axis}") 
        
    return full_columns

ALL_COLUMNS = generate_full_column_list()

In [24]:
def normalize_values(df):
    axes = ['x', 'y', 'z'] if INCLUDE_DEPTH else ['x', 'y']

    origins = (
        df[(df['type'] == 'pose') & (df['landmark_index'] == 0)]
        .set_index('frame')[axes]
    )

    def normalize_frame(frame_df):
        frame = frame_df.name
        if frame not in origins.index:
            return frame_df  # or raise an error
        frame_df[axes] = frame_df[axes] - origins.loc[frame]
        return frame_df

    return df.groupby('frame', group_keys=False).apply(normalize_frame)

def frame_stacked_data(file_path):
    df = pd.read_parquet(file_path)
    if not INCLUDE_FACE:
        df = df[df['type'] != 'face']
    
    if not INCLUDE_DEPTH:
        df.drop('z', axis=1, inplace=True)
    axes = ['x', 'y', 'z'] if INCLUDE_DEPTH else ['x', 'y']

    df['UID'] = df['type'] + '_' + df['landmark_index'].astype('str')

    df = df.sort_values(['frame', 'UID'])
    df = normalize_values(df)

    pivot_df = df.pivot_table(index='frame', columns='UID', values=axes)
    pivot_df.columns = [f"{col[1]}_{col[0]}" for col in pivot_df.columns]
    pivot_df = pivot_df.reindex(columns=ALL_COLUMNS)

    final_data = pd.DataFrame(pivot_df).ffill().bfill().fillna(0).to_numpy()
    return final_data

In [25]:
data_file_path = r"../Kaggle_Data/train_landmark_files/16069/100015657.parquet"
data = frame_stacked_data(data_file_path)

/var/folders/gj/hbctk6fn5k75zhf2xtxjhz880000gn/T/ipykernel_20960/3737309998.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('frame', group_keys=False).apply(normalize_frame)


In [19]:
data

array([[0.43675497, 0.29740363, 0.34558299, ..., 0.        , 0.        ,
        0.        ],
       [0.41852686, 0.29012206, 0.33899215, ..., 0.        , 0.        ,
        0.        ],
       [0.40473375, 0.28120434, 0.32387909, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.45109895, 0.07582352, 0.37355718, ..., 0.        , 0.        ,
        0.        ],
       [0.43803057, 0.15679425, 0.37280342, ..., 0.        , 0.        ,
        0.        ],
       [0.41130546, 0.16640994, 0.36008921, ..., 0.        , 0.        ,
        0.        ]], shape=(105, 150))

In [2]:
data_file_path = r"../Kaggle_Data/train_landmark_files/16069/100015657.parquet"
data_df = pd.read_parquet(data_file_path)

In [11]:
data_df.head()

,frame,row_id,type,landmark_index,x,y,z,UID
0,103,103-face-0,face,0,0.437886,0.437599,-0.051134,face0
1,103,103-face-1,face,1,0.443258,0.392901,-0.067054,face1
2,103,103-face-2,face,2,0.443997,0.409998,-0.042990,face2
3,103,103-face-3,face,3,0.435256,0.362771,-0.039492,face3
4,103,103-face-4,face,4,0.443780,0.381762,-0.068013,face4


In [ ]:
data_df['UID'] = data_df['type'] + data_df['landmark_index'].astype('str')

In [63]:
data_df = data_df.sort_values(['frame', 'UID'])
required_columns = data_df['UID'].unique().tolist()

full_columns = [
    f"{col}_{axis}"
    for col in required_columns
    for axis in ['x', 'y', 'z']
]

In [67]:
pivot_data = data_df.pivot_table(index='frame', columns='UID', values=['x', 'y', 'z'])
pivot_data

x                                                              \
UID      face_0    face_1   face_10  face_100  face_101  face_102  face_103   
frame                                                                         
103    0.437886  0.443258  0.452921  0.392363  0.370901  0.401237  0.350024   
104    0.435957  0.444225  0.460122  0.392338  0.369927  0.398563  0.352889   
105    0.438280  0.443547  0.455426  0.390452  0.368441  0.398290  0.350066   
106    0.436610  0.440847  0.455194  0.389891  0.367716  0.396508  0.350160   
107    0.439554  0.441069  0.454005  0.389369  0.367044  0.396702  0.348616   
...         ...       ...       ...       ...       ...       ...       ...   
203    0.413113  0.402174  0.419720  0.357568  0.335687  0.363685  0.315194   
204    0.409165  0.403373  0.419655  0.357406  0.335622  0.363794  0.315019   
205    0.413181  0.401993  0.416854  0.356982  0.335345  0.363110  0.313388   
206    0.414697  0.402356  0.417589  0.357058  0.335289  0.363040  0.313902   
207    0.409994  0.401792  0.417536  0.356519  0.334778  0.362630  0.313474   

                                     ...         z                      \
UID    face_104  face_105  face_106  ...    pose_3   pose_30   pose_31   
frame                                ...                                 
103    0.355997  0.364375  0.392658  ... -1.391819  1.109108  0.458282   
104    0.358186  0.364776  0.386953  ... -1.412054  1.096976  0.476854   
105    0.356060  0.363025  0.385209  ... -1.414343  1.058633  0.453952   
106    0.355356  0.360959  0.385647  ... -1.315845  0.939371  0.321508   
107    0.354757  0.358172  0.385838  ... -1.337843  0.805818  0.157870   
...         ...       ...       ...  ...       ...       ...       ...   
203    0.319826  0.324342  0.356932  ... -1.160070  0.735054  0.254781   
204    0.319941  0.323297  0.356781  ... -1.252138  0.743792  0.260332   
205    0.318039  0.320053  0.357812  ... -1.287089  0.823286  0.300238   
206    0.318601  0.321648  0.357809  ... -1.300317  0.854767  0.308823   
207    0.318047  0.321158  0.357129  ... -1.320961  0.864832  0.313457   

                                                                             
UID     pose_32    pose_4    pose_5    pose_6    pose_7    pose_8    pose_9  
frame                                                                        
103    0.077913 -1.401795 -1.400807 -1.401427 -0.764340 -0.795864 -1.251937  
104    0.050753 -1.425607 -1.424708 -1.425366 -0.772132 -0.818006 -1.273108  
105    0.012233 -1.425760 -1.424871 -1.425520 -0.778715 -0.817601 -1.276169  
106   -0.078689 -1.323410 -1.322383 -1.322988 -0.712748 -0.734179 -1.185205  
107   -0.253118 -1.350978 -1.349783 -1.349980 -0.719245 -0.768183 -1.223318  
...         ...       ...       ...       ...       ...       ...       ...  
203   -0.145288 -1.183443 -1.183150 -1.184111 -0.640904 -0.724059 -1.036942  
204   -0.144940 -1.263117 -1.262867 -1.264037 -0.705543 -0.746628 -1.124535  
205   -0.107871 -1.300119 -1.299734 -1.300898 -0.703584 -0.748627 -1.155408  
206   -0.088305 -1.316249 -1.315814 -1.316947 -0.701278 -0.749589 -1.167087  
207   -0.082251 -1.339295 -1.338847 -1.339970 -0.704033 -0.760024 -1.186551  

[105 rows x 1566 columns]

In [65]:
# pivot_data.columns = [f'{coord}_{uid}' for coord, uid in pivot_data.columns]
pivot_data = pivot_data.reindex(columns=full_columns)

In [66]:
flat_data = pivot_data.ffill().bfill().fillna(0).to_numpy()
flat_data.shape

(105, 1629)

In [48]:
pivot_data.to_numpy().shape

(105, 57015)

In [33]:
def flatten_data(file_path):
    df = pd.read_parquet(file_path)
    df['UID'] = df['type'] + df['landmark_index'].astype('str')

    pivot_data = data_df.pivot_table(index='frame', columns='UID', values=['x', 'y', 'z'])
    flat_data = pd.DataFrame(pivot_data).ffill().bfill().fillna(0).to_numpy()
    
    return flat_data

In [34]:
class ASLDataset:
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        flat_data = flatten_data(self.file_paths[idx])
        return torch.tensor(flat_data)

In [35]:
import torch.nn as nn
import torch

class ASLAutoencoder(nn.Module):
    def __init__(self, input_size):
        super(ASLAutoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_size)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        
        return decoded


In [36]:
import torch.optim as optim

model = ASLAutoencoder(1566)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [43]:
data_torch = torch.tensor(flatten_data(r"../Kaggle_Data/train_landmark_files/16069/100015657.parquet")).float()
test_torch = torch.tensor(flatten_data(r"../Kaggle_Data/train_landmark_files/18796/1001373962.parquet")).float()

In [42]:
model.train()


for _ in range(10):
    outputs = model(data_torch)
    loss = criterion(outputs, data_torch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(loss.item())

0.1953209787607193
0.18410509824752808
0.1642991602420807
0.11353698372840881
0.056575242429971695
0.05543017387390137
0.03152580186724663
0.03972860798239708
0.030774157494306564
0.02128095179796219


In [44]:
model.eval()

output = model(test_torch)
loss = criterion(output, test_torch)

loss.item()

0.031567540019750595